# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaihanBasha7/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — “AI-generated content is penalized by default” is false in the studied dataset

The report compares outcomes across AI-authored content cohorts and concludes that it does not show a blanket penalty tied only to AI use.

**My methodology question:** Where does the label/cohort definition come from, and how consistently was AI authorship identified? I would also ask whether model cohorts differ in topic, content age, editing workflow, or publication period. The comparison is useful for describing an observed pattern in this dataset, but those differences could still affect the result. I would therefore avoid extending the finding into a universal claim that AI-generated content is never penalized.

### Finding 2 — Refreshing mature pages is associated with stronger outcomes

The report compares older pages that were refreshed recently with older pages that were not recently refreshed and reports large differences in health and impressions. It also reports a held-out refresh analysis with statistically significant effects across most strata.

**My methodology question:** How were pages selected for refresh, and were refreshed pages already different from stale pages before the refresh? For example, they may have had more historical visibility, strategic value, or stronger baseline performance. Even with a held-out comparison, selection effects can remain. The evidence supports an observed/directional association and an operational hypothesis that refreshing mature pages can help, but I would be careful about claiming that the refresh itself caused every reported lift without a stronger causal design.

In [21]:
# Setup and data load
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Week-5 label definition
TARGET = "is_declining_label"
if TARGET not in df.columns:
    df[TARGET] = (
        df["trend_direction"].astype(str).str.lower().eq("down")
    ).astype(int)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Clients: {df['client_id'].nunique():,}")
print(f"Declining base rate: {df[TARGET].mean():.3f} ({df[TARGET].mean()*100:.1f}%)")

Rows: 30,000
Columns: 45
Clients: 32
Declining base rate: 0.542 (54.2%)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model is a Random Forest ranking/classification model. The model uses the same six Week-5 predictors:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`
- `word_count`

For the **before** result, I intentionally use a naive row-level split. This is not the preferred deployment estimate because pages from the same client can appear in both train and test.

For the **after** result, I use a client-grouped split. No client is allowed to appear in both sets. This better tests the question: **does the model generalize to a client it has not seen during training?**

The primary ranking metrics are Precision@20 and Precision@50, matching the Week-5 review workflow. Accuracy/F1 are also reported as secondary classification metrics.

In [22]:
# Recreate the Week-5 feature set.
W05_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

missing_features = [c for c in W05_FEATURES if c not in df.columns]
if missing_features:
    raise KeyError(f"Week-5 feature(s) missing from dataset: {missing_features}")

X = df[W05_FEATURES].copy()
y = df[TARGET].astype(int)
groups = df["client_id"]

print("Week-5 features:")
print(W05_FEATURES)
print("\nBase rate:", round(y.mean(), 4))

Week-5 features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Base rate: 0.5421


In [23]:
# Random Forest pipeline.
# Median imputation is fitted inside each training split, so the test set
# does not influence preprocessing.

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def evaluate(train_idx, test_idx, split_name):
    model = make_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    pred = (scores >= 0.5).astype(int)

    y_test = y.iloc[test_idx].to_numpy()

    return {
        "Split": split_name,
        "Train rows": len(train_idx),
        "Test rows": len(test_idx),
        "Test clients": groups.iloc[test_idx].nunique(),
        "Positive rate": y_test.mean(),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
    }, model, scores, y_test

In [24]:
# Verify the grouped split is genuinely client-holdout.

from sklearn.model_selection import train_test_split, GroupShuffleSplit

# ============================================================
# 1. BEFORE — naive random row-level split
# ============================================================

row_train_idx, row_test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

row_result, row_model, row_scores, row_y_test = evaluate(
    row_train_idx,
    row_test_idx,
    "Before — random row split"
)

# ============================================================
# 2. AFTER — honest client-grouped split
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

group_train_idx, group_test_idx = next(
    gss.split(X, y, groups=groups)
)

group_result, group_model, group_scores, group_y_test = evaluate(
    group_train_idx,
    group_test_idx,
    "After — grouped by client"
)

# ============================================================
# 3. Verify zero client overlap
# ============================================================

train_clients = set(groups.iloc[group_train_idx])
test_clients = set(groups.iloc[group_test_idx])

client_overlap = train_clients.intersection(test_clients)

print("Grouped train clients:", len(train_clients))
print("Grouped test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0, \
    "FAIL: client leakage exists in grouped split."

print("PASS: grouped split has zero client overlap.")

# ============================================================
# 4. BEFORE vs AFTER comparison
# ============================================================

before_after = pd.DataFrame({
    "Metric": [
        "Precision@20",
        "Precision@50",
        "Accuracy",
        "F1"
    ],

    "Before — random row": [
        row_result["Precision@20"],
        row_result["Precision@50"],
        row_result["Accuracy"],
        row_result["F1"]
    ],

    "After — grouped client": [
        group_result["Precision@20"],
        group_result["Precision@50"],
        group_result["Accuracy"],
        group_result["F1"]
    ]
})

before_after["Change (after - before)"] = (
    before_after["After — grouped client"]
    - before_after["Before — random row"]
)

display(before_after.round(4))

Grouped train clients: 25
Grouped test clients: 7
Client overlap: 0
PASS: grouped split has zero client overlap.


,Metric,Before — random row,After — grouped client,Change (after - before)
0,Precision@20,0.8500,0.8500,0.0000
1,Precision@50,0.8800,0.7000,-0.1800
2,Accuracy,0.6835,0.5629,-0.1206
3,F1,0.7192,0.5928,-0.1264


### Interpretation of the validation gap

The random split is useful as a diagnostic, but the grouped-by-client result is the more honest estimate for an unseen-client use case.

If the grouped metrics are lower, that gap is itself a finding: a random split can make performance look more optimistic when rows from the same client are present in both train and test.

I therefore report both numbers, but I use the grouped result when making the main generalization claim. I do not interpret a lower grouped score as a failure of the model; it is evidence about how much performance depends on client-specific information.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The leakage audit follows three checks:

1. **Label-derived features:** `trend_direction` creates the label through `trend_pct`, so neither field may be a feature.
2. **Future/overlapping windows:** a feature must be available before the prediction point and must not contain the outcome window.
3. **Decision-derived/identifier features:** existing decision flags, scores, `client_id`, and `content_id` should not be predictive inputs.

The Week-5 starter dataset uses trailing-90-day metrics, so the exact timing of those metrics must be treated carefully. This notebook therefore documents the feature-level exclusion checks and does not claim that a simple column-name audit proves perfect temporal causality.

In [25]:
# Explicit feature leakage audit
FORBIDDEN_FEATURES = {
    "trend_direction": "Directly used to define the target",
    "trend_pct": "Source field used to compute trend_direction",
    "client_id": "Grouping identifier; must not be a predictive feature",
    "content_id": "Row/content identifier; must not be a predictive feature",
    TARGET: "Target itself",
}

audit_rows = []

for column, reason in FORBIDDEN_FEATURES.items():
    audit_rows.append({
        "Column": column,
        "Present in model features?": column in W05_FEATURES,
        "Allowed?": "NO",
        "Reason": reason,
    })

leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)

violations = [
    row["Column"]
    for _, row in leakage_audit.iterrows()
    if row["Present in model features?"]
]

print("Known leakage violations:", violations)
assert not violations, f"FAIL: forbidden feature(s) detected: {violations}"
print("PASS: no known label-derived, ID, or target columns are model features.")

,Column,Present in model features?,Allowed?,Reason
0,trend_direction,False,NO,Directly used to define the target
1,trend_pct,False,NO,Source field used to compute trend_direction
2,client_id,False,NO,Grouping identifier; must not be a predictive ...
3,content_id,False,NO,Row/content identifier; must not be a predicti...
4,is_declining_label,False,NO,Target itself


Known leakage violations: []
PASS: no known label-derived, ID, or target columns are model features.


In [26]:
# Timeline / availability audit for the six Week-5 features.
# This is a documentation check: all six fields must represent information
# available before the model's decision point.

availability_audit = pd.DataFrame([
    {
        "Feature": "content_age_days",
        "Timing check": "Content attribute known at prediction time",
        "Status": "PASS",
    },
    {
        "Feature": "days_since_last_update",
        "Timing check": "Update recency known at prediction time",
        "Status": "PASS",
    },
    {
        "Feature": "impressions_90d",
        "Timing check": "Historical performance feature; window must end before outcome window",
        "Status": "REVIEWED",
    },
    {
        "Feature": "avg_position",
        "Timing check": "Historical search-position feature; window must end before outcome window",
        "Status": "REVIEWED",
    },
    {
        "Feature": "ctr",
        "Timing check": "Historical CTR feature; window must end before outcome window",
        "Status": "REVIEWED",
    },
    {
        "Feature": "word_count",
        "Timing check": "Content attribute known at prediction time",
        "Status": "PASS",
    },
])

display(availability_audit)

,Feature,Timing check,Status
0,content_age_days,Content attribute known at prediction time,PASS
1,days_since_last_update,Update recency known at prediction time,PASS
2,impressions_90d,Historical performance feature; window must en...,REVIEWED
3,avg_position,Historical search-position feature; window mus...,REVIEWED
4,ctr,Historical CTR feature; window must end before...,REVIEWED
5,word_count,Content attribute known at prediction time,PASS


In [27]:
# Deliberate leakage test: add trend_pct, which is forbidden.
# The purpose is educational: a suspiciously strong result demonstrates why
# the clean model must exclude label-derived information.

if "trend_pct" in df.columns:
    X_leaky = df[W05_FEATURES + ["trend_pct"]].copy()

    leaky_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    leaky_model.fit(X_leaky.iloc[group_train_idx], y.iloc[group_train_idx])
    leaky_scores = leaky_model.predict_proba(
        X_leaky.iloc[group_test_idx]
    )[:, 1]

    leaky_p50 = precision_at_k(
        y.iloc[group_test_idx].to_numpy(),
        leaky_scores,
        50
    )

    print("Leaky Precision@50:", round(leaky_p50, 4))
    print("Clean grouped Precision@50:", round(group_result["Precision@50"], 4))
    print("\nThis is a diagnostic only. trend_pct MUST NOT be used in the final model.")
else:
    print("trend_pct column not found; skipped deliberate leakage demonstration.")

Leaky Precision@50: 1.0
Clean grouped Precision@50: 0.7

This is a diagnostic only. trend_pct MUST NOT be used in the final model.


### Leakage conclusion

The final Week-5 feature set contains none of the known target-source fields or identifiers. The deliberate `trend_pct` experiment, when available, demonstrates why this matters: target-derived information can create an artificially strong ranking signal.

The remaining limitation is temporal: the starter dataset's historical metric windows must be interpreted relative to the exact label window. I therefore describe the audit as a documented leakage check, not as proof that every possible temporal dependency has been eliminated.

### Real failure examples from the honest split

The following examples are taken only from the grouped test set. They show cases where the model's binary prediction disagreed with the observed label.

I inspect both false positives and false negatives because a model can fail in different ways. These examples are used for diagnosis, not to claim that the model has a particular root cause without further testing.

In [28]:
# Build an error table for the grouped test set.
group_test = df.iloc[group_test_idx].copy()
group_test["predicted_probability"] = group_scores
group_test["predicted_label"] = (group_scores >= 0.5).astype(int)
group_test["error_type"] = np.select(
    [
        (group_test[TARGET] == 0) & (group_test["predicted_label"] == 1),
        (group_test[TARGET] == 1) & (group_test["predicted_label"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct"
)

failure_columns = [
    c for c in [
        "content_id",
        "client_id",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "word_count",
        TARGET,
        "predicted_probability",
        "predicted_label",
        "error_type",
    ]
    if c in group_test.columns
]

failures = group_test[group_test["error_type"] != "Correct"]

print("Grouped-test failures:", len(failures))
display(
    failures[failure_columns]
    .sort_values("predicted_probability")
    .head(10)
)

Grouped-test failures: 2694


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label,predicted_probability,predicted_label,error_type
28032,content_b2beaf2fc81c,client_e629fa6598,502,22,767,53.4,0.00,NaN,1,0.020,0,False negative
13114,content_8f222654e93f,client_8527a891e2,273,20,2,5.5,0.00,1994.0,1,0.020,0,False negative
23815,content_df23b4bc766d,client_e629fa6598,502,22,665,49.6,0.00,NaN,1,0.020,0,False negative
252,content_aba4b4460e47,client_e629fa6598,460,22,1334,50.1,0.00,NaN,1,0.020,0,False negative
29729,content_dc67ab936666,client_e629fa6598,460,22,3396,31.7,0.06,NaN,1,0.025,0,False negative
2549,content_0a4a2b15186e,client_e629fa6598,460,22,482,35.3,0.21,NaN,1,0.030,0,False negative
10449,content_e3a34c1706cf,client_8527a891e2,127,104,2,5.0,0.00,4074.0,1,0.040,0,False negative
21667,content_af59c0a2deb3,client_e629fa6598,460,22,44,25.8,0.00,NaN,1,0.040,0,False negative
27248,content_a2a9d1ffeca3,client_e629fa6598,460,22,1065,23.9,0.28,NaN,1,0.040,0,False negative
9961,content_17d206dcc70b,client_e629fa6598,490,22,1947,35.8,0.05,NaN,1,0.055,0,False negative


In [29]:
# Error counts and rates
error_summary = pd.DataFrame({
    "Error type": ["False positives", "False negatives"],
    "Count": [
        int(((group_test[TARGET] == 0) & (group_test["predicted_label"] == 1)).sum()),
        int(((group_test[TARGET] == 1) & (group_test["predicted_label"] == 0)).sum()),
    ]
})

error_summary["Rate of grouped test"] = (
    error_summary["Count"] / len(group_test)
)

display(error_summary.round(4))

,Error type,Count,Rate of grouped test
0,False positives,1506,0.2444
1,False negatives,1188,0.1928


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### My original bold claim

> **“The Random Forest can reliably identify declining content and tell us which pages should be refreshed.”**

### Evidence-aware rewrite

> **Observed:** The Random Forest produced a useful ranking signal for declining content in the evaluated dataset. **Measured:** its Precision@20 and Precision@50 can be compared under both a naive row-level split and a client-grouped split, with the grouped result providing the more conservative estimate for unseen-client generalization. **Directional:** the results suggest the model may help prioritize pages that deserve review. **Decision-support:** the score should be used as a prioritization aid alongside human review, not as proof that a page needs a refresh or that a refresh will improve performance.\

In [30]:
# Evidence table supporting the rewritten claim
claim_evidence = pd.DataFrame({
    "Evidence": [
        "Declining base rate",
        "Random-split Precision@50",
        "Grouped-client Precision@50",
        "Client overlap in honest split",
        "Known leakage violations",
    ],
    "Measured value": [
        y.mean(),
        row_result["Precision@50"],
        group_result["Precision@50"],
        len(client_overlap),
        len(violations),
    ]
})

display(claim_evidence.round(4))

print(
    "Claim discipline: report the measured values above and avoid "
    "language such as 'guarantees', 'always', 'reliably', or 'causes'."
)

,Evidence,Measured value
0,Declining base rate,0.5421
1,Random-split Precision@50,0.8800
2,Grouped-client Precision@50,0.7000
3,Client overlap in honest split,0.0000
4,Known leakage violations,0.0000


Claim discipline: report the measured values above and avoid language such as 'guarantees', 'always', 'reliably', or 'causes'.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.